# 03 - LLM Extraction Experiment

**Research question:** Does task decomposition improve hazard-information extraction compared with a single end-to-end LLM prompt?

Both workflows use the same model, schemas, cleaned Markdown, and paragraph-aware chunks. Workflow A makes one end-to-end call per chunk; Workflow B makes four task-specific calls per chunk. The original full-document pilot remains unchanged in `data/results/`.

## 1. Setup

In [1]:
# Import the small set of libraries used for paths, chunking, JSON, and timing.
from pathlib import Path
import bisect
import json
import re
import sys
import time

import pandas as pd

# Find the project folder whether the notebook is launched from the project root or notebooks/.
PROJECT_FOLDER = Path.cwd().resolve()
for candidate in [PROJECT_FOLDER, PROJECT_FOLDER.parent]:
    if (candidate / "data/evaluation/manifest.json").exists():
        PROJECT_FOLDER = candidate
        break
else:
    raise FileNotFoundError("Could not find data/evaluation/manifest.json")

sys.path.insert(0, str(PROJECT_FOLDER / "src"))

# Reuse the common Pydantic schemas and deterministic merge helper.
from extraction_schema import (
    DocumentExtraction,
    HazardTaskOutput,
    ImpactTaskOutput,
    LocationTaskOutput,
    TimeTaskOutput,
)
from workflows import merge_decomposed_outputs

# Frozen settings used for both workflows in the final experiment.
MODEL = "openai/gpt-oss-20b"
TEMPERATURE = 0.01
SEED = 42
MAX_OUTPUT_TOKENS = 4096
TIMEOUT_SECONDS = 180
REASONING_EFFORT = "low"
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
CONTEXT_LIMIT = 128_000
MAX_CHUNK_WORDS = 800
CHUNK_OVERLAP_WORDS = 100
# Keep this False when reviewing the notebook to prevent NVIDIA API calls.
RUN_LLM = False

# All inputs and outputs use repository-relative paths.
MANIFEST_PATH = PROJECT_FOLDER / "data/evaluation/manifest.json"
DOCUMENT_DIR = PROJECT_FOLDER / "data/evaluation/documents"
RESULTS_DIR = PROJECT_FOLDER / "data/final_results"

print(f"Project folder: {PROJECT_FOLDER}")
print(
    f"Model: {MODEL}; temperature={TEMPERATURE}; seed={SEED}; "
    f"max output={MAX_OUTPUT_TOKENS}; reasoning_effort={REASONING_EFFORT}"
)
print(f"Chunks: maximum {MAX_CHUNK_WORDS} words; overlap {CHUNK_OVERLAP_WORDS} words")
print(f"RUN_LLM={RUN_LLM}")

Project folder: C:\Users\onssr\Desktop\AIHWS\text2hazard_project
Model: openai/gpt-oss-20b; temperature=0.01; seed=42; max output=4096; reasoning_effort=low
Chunks: maximum 800 words; overlap 100 words
RUN_LLM=False


In [ ]:
# NVIDIA API setup adapted from the course exercise (Session 14)
# Importing the API client is delayed until RUN_LLM=True, so review execution does not require a token.
# The shared client in src/llm_client.py applies the frozen model settings,so both workflows use exactly the same NVIDIA configuration.
def create_experiment_llm():
    from llm_client import create_llm

    return create_llm(MODEL)


# Send one prompt and require the response to follow the supplied Pydantic schema.
def ask_for_structure(llm, output_schema, prompt):
    from langchain_core.callbacks import BaseCallbackHandler
    from langchain_core.messages import HumanMessage, SystemMessage

    # Capture NVIDIA's finish_reason from the real response metadata.
    # This makes it possible to distinguish a complete response from one that stopped because it reached the output-token limit.
    class ResponseMetadataHandler(BaseCallbackHandler):
        def __init__(self):
            super().__init__()
            self.finish_reason = None

        def on_llm_end(self, response, **kwargs):
            try:
                message = response.generations[0][0].message
                self.finish_reason = message.response_metadata.get("finish_reason")
            except (AttributeError, IndexError, TypeError):
                self.finish_reason = None

    # LangChain converts the Pydantic class into the structured-output request.
    metadata_handler = ResponseMetadataHandler()
    structured_llm = llm.with_structured_output(output_schema)
    # The common system instructions and the workflow-specific task prompt are passed as separate chat messages, following the course example.
    parsed = structured_llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=prompt),
    ], config={"callbacks": [metadata_handler]})

    # Truncated, metadata-free, or unparsable responses are recorded as failures instead of being accepted as successful extractions.
    if metadata_handler.finish_reason in {"length", "max_tokens"}:
        raise RuntimeError("Model response was truncated at the output-token limit.")
    if metadata_handler.finish_reason is None:
        raise RuntimeError("NVIDIA response did not include finish_reason metadata.")
    if parsed is None:
        raise ValueError("Model returned no valid structured response.")
    # Validate once more before returning the typed extraction object.
    return output_schema.model_validate(parsed)

## 2. Evaluation documents

Only the existing manifest and its nine cleaned Markdown files are loaded. Reference annotations are intentionally not read in this notebook.

In [3]:
WORD_PATTERN = r"\b[^\W\d_]+(?:['’\-][^\W\d_]+)*\b"


# Use one consistent Unicode-aware word definition for statistics and chunking.
def count_words(text):
    return len(re.findall(WORD_PATTERN, text, flags=re.UNICODE))


# Documents up to 800 words remain intact. Longer texts use a 100-word
# overlap, while nearby paragraph endings are preferred as boundaries.
def split_into_chunks(text, max_words=MAX_CHUNK_WORDS, overlap_words=CHUNK_OVERLAP_WORDS):
    """Split text by words, preferring nearby Markdown paragraph boundaries."""
    words = list(re.finditer(WORD_PATTERN, text, flags=re.UNICODE))
    if len(words) <= max_words:
        return [{"chunk_id": "chunk_001", "text": text, "word_count": len(words)}]

    word_ends = [match.end() for match in words]
    paragraph_word_ends = [
        bisect.bisect_right(word_ends, match.start())
        for match in re.finditer(r"\n\s*\n", text)
    ]

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_words, len(words))
        if end < len(words):
            minimum_boundary = start + max_words - overlap_words
            candidates = [
                boundary
                for boundary in paragraph_word_ends
                if minimum_boundary <= boundary <= end
            ]
            if candidates:
                end = candidates[-1]

        start_char = 0 if start == 0 else words[start].start()
        end_char = len(text) if end == len(words) else words[end].start()
        chunk_text = text[start_char:end_char].strip()
        chunks.append({
            "chunk_id": f"chunk_{len(chunks) + 1:03d}",
            "text": chunk_text,
            "word_count": count_words(chunk_text),
        })

        if end == len(words):
            break
        start = end - overlap_words

    return chunks


# Load exactly the nine frozen evaluation documents listed in the manifest.
# Reference annotations are not opened anywhere in this extraction notebook.
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
assert len(manifest["documents"]) == 9, "The evaluation manifest must contain exactly nine documents."

documents = []
for item in manifest["documents"]:
    path = DOCUMENT_DIR / item["filename"]
    text = path.read_text(encoding="utf-8")
    chunks = split_into_chunks(text)
    documents.append({
        "document_id": item["document_id"],
        "source_type": item["document_type"],
        "word_count": count_words(text),
        "chunk_count": len(chunks),
        "filename": item["filename"],
        "text": text,
        "chunks": chunks,
    })

document_table = pd.DataFrame(documents)[
    ["document_id", "source_type", "word_count", "chunk_count"]
]
display(document_table)
print(f"Loaded {len(documents)} cleaned Markdown documents.")

,document_id,source_type,word_count,chunk_count
0,PWC 2015 - Updated estimates on cost of Storm ...,Institutional / technical report,484,1
1,Krausmann 2014 - STREST report on lessons lear...,Institutional / technical report,1469,2
2,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,Institutional / technical report,10477,16
3,Lavenir 2021 - À la recherche d'une jeune fill...,News / media,157,1
4,ABC 2024 - Traffic jams and flight delays due ...,News / media,706,1
5,"EFE 2024 - The DANA storm, live_ The death tol...",News / media,3635,6
6,Plag 2014 - Foreword extreme geohazards—a grow...,Scientific paper,1457,3
7,Kadir 2014 - The Impact of Natural Disasters o...,Scientific paper,4935,8
8,"Cavaleri 2019 - The October 29, 2018 storm in ...",Scientific paper,9218,14


Loaded 9 cleaned Markdown documents.


## 3. Extraction schema

Both workflows produce the existing `DocumentExtraction` schema. A document contains zero or more events; each event has at least one supported hazard and lists its locations, times, and impacts. Every fact carries a verbatim evidence quote. The decomposed task schemas contain the same field models, so their outputs can be merged without changing the target representation.


In [4]:
# Show the common document-level schema produced by both workflows.
schema_overview = pd.DataFrame([
    {"level": "document", "fields": "document_id, events, notes"},
    {"level": "event", "fields": "event_id, hazards, locations, times, impacts"},
    {"level": "hazard", "fields": "category, source_text, evidence.quote"},
    {"level": "location", "fields": "source_text, normalized_name, country_code, evidence.quote"},
    {"level": "time", "fields": "source_text, normalized_start, normalized_end, evidence.quote"},
    {"level": "impact", "fields": "category, description, evidence.quote"},
])
display(schema_overview)

,level,fields
0,document,"document_id, events, notes"
1,event,"event_id, hazards, locations, times, impacts"
2,hazard,"category, source_text, evidence.quote"
3,location,"source_text, normalized_name, country_code, ev..."
4,time,"source_text, normalized_start, normalized_end,..."
5,impact,"category, description, evidence.quote"


In [5]:
# These instructions define the extraction rules shared by every API call.
SYSTEM_PROMPT = """You extract structured natural-hazard information from one supplied document.
Use only the document: do not use outside knowledge or guess missing values.
Extract events actually described in the document. Exclude passing comparisons, citations, hypothetical events, projections, and generic background statements.
Related hazards in one episode may belong to the same event, and repeated mentions must not create duplicate events.
Locations must be explicitly connected to the event. Normalize time conservatively and never infer a calendar date from a publication date.
Impacts must be concrete consequences. Preserve numbers, units, currencies, and qualifiers.
Every extracted fact must contain a short verbatim evidence quote from the cleaned Markdown.
Return only structured output matching the supplied Pydantic schema."""


# Mark the source text clearly so it remains separate from the instructions.
def document_block(document_id, text):
    return f"DOCUMENT_ID: {document_id}\n\n<DOCUMENT>\n{text}\n</DOCUMENT>"

## 4. Workflow A - Single prompt per chunk

Workflow A applies the existing end-to-end prompt independently to every chunk. The validated chunk outputs are combined with the same conservative document-level merge used after Workflow B.

In [6]:
# Workflow A requests the complete extraction in one LLM call per chunk.
def build_workflow_a_prompt(document_id, text):
    return f"""Extract all qualifying natural-hazard events from the document.
For each event, extract its hazards, explicitly connected locations, supported times, concrete impacts, and verbatim evidence for every fact.
Assign stable document-local IDs event_01, event_02, and so on. Use empty lists for unsupported optional fields and set notes to an empty list.

{document_block(document_id, text)}"""


# Parse the response directly into the common document-level schema.
def run_workflow_a(llm, document_id, text):
    return ask_for_structure(
        llm,
        DocumentExtraction,
        build_workflow_a_prompt(document_id, text),
    )

## 5. Workflow B - Decomposed prompts per chunk

For each chunk, Step 1 identifies its event catalogue and hazards. Steps 2-4 receive that catalogue and the same chunk text to extract locations, times, and impacts. The existing within-chunk merge is followed by the same document-level merge used for Workflow A.

In [7]:
# Workflow B stage 1 creates the event catalogue and extracts hazards only.
def build_hazard_prompt(document_id, text):
    return f"""Identify all qualifying natural-hazard events and extract only their hazards and evidence.
Assign stable document-local IDs event_01, event_02, and so on. Return no events if the document contains only excluded background, comparisons, citations, hypotheses, or projections.
Do not extract locations, times, or impacts in this step.

{document_block(document_id, text)}"""


# Stages 2-4 receive the same chunk plus the stage-1 event catalogue.
# They may enrich listed events but cannot create new events.
def build_enrichment_prompt(field_name, document_id, text, hazard_events):
    event_catalogue = hazard_events.model_dump_json(indent=2)
    return f"""Extract only {field_name} for the events already identified in EVENT CATALOGUE.
Return every supplied event_id exactly once. Do not create, remove, merge, split, or rename events.
Use an empty list when {field_name} are not supported for an event. Include verbatim evidence for every extracted fact.

EVENT CATALOGUE:
{event_catalogue}

{document_block(document_id, text)}"""


# These wrappers keep the enrichment prompts identical except for the field.
def build_location_prompt(document_id, text, hazards):
    return build_enrichment_prompt("locations", document_id, text, hazards)


def build_time_prompt(document_id, text, hazards):
    return build_enrichment_prompt("times", document_id, text, hazards)


def build_impact_prompt(document_id, text, hazards):
    return build_enrichment_prompt("impacts", document_id, text, hazards)

In [8]:
# Run the four Workflow B calls in order and merge their outputs by event_id.
def run_workflow_b(llm, document_id, text):
    hazards = ask_for_structure(
        llm, HazardTaskOutput, build_hazard_prompt(document_id, text)
    )
    locations = ask_for_structure(
        llm, LocationTaskOutput, build_location_prompt(document_id, text, hazards)
    )
    times = ask_for_structure(
        llm, TimeTaskOutput, build_time_prompt(document_id, text, hazards)
    )
    impacts = ask_for_structure(
        llm, ImpactTaskOutput, build_impact_prompt(document_id, text, hazards)
    )
    final = merge_decomposed_outputs(hazards, locations, times, impacts)
    return hazards, locations, times, impacts, final


# The following helpers support conservative deterministic cross-chunk merging.
def _normalized_quotes(event):
    facts = event.hazards + event.locations + event.times + event.impacts
    return {
        " ".join(fact.evidence.quote.casefold().split())
        for fact in facts
    }


def _hazard_categories(event):
    return {hazard.category.value for hazard in event.hazards}


def _extend_unique(target_items, source_items):
    seen = {
        json.dumps(item.model_dump(mode="json"), sort_keys=True, ensure_ascii=False)
        for item in target_items
    }
    for item in source_items:
        key = json.dumps(item.model_dump(mode="json"), sort_keys=True, ensure_ascii=False)
        if key not in seen:
            target_items.append(item.model_copy(deep=True))
            seen.add(key)


# Merge only adjacent-chunk events sharing both an evidence quote and hazard category.
# All other events remain separate rather than being matched semantically.
def merge_chunk_extractions(document_id, chunk_outputs):
    """Merge adjacent-chunk duplicates using exact evidence and hazard overlap."""
    merged = []
    notes = []

    for chunk_number, output in enumerate(chunk_outputs, start=1):
        if output.document_id != document_id:
            raise ValueError("A chunk output has the wrong document_id.")

        for note in output.notes:
            if note not in notes:
                notes.append(note)

        for event in output.events:
            match = None
            for candidate in merged:
                if candidate["last_chunk"] != chunk_number - 1:
                    continue
                existing = candidate["event"]
                if (
                    _normalized_quotes(existing) & _normalized_quotes(event)
                    and _hazard_categories(existing) & _hazard_categories(event)
                ):
                    match = candidate
                    break

            if match is None:
                merged.append({
                    "event": event.model_copy(deep=True),
                    "last_chunk": chunk_number,
                })
            else:
                existing = match["event"]
                _extend_unique(existing.hazards, event.hazards)
                _extend_unique(existing.locations, event.locations)
                _extend_unique(existing.times, event.times)
                _extend_unique(existing.impacts, event.impacts)
                match["last_chunk"] = chunk_number

    # Assign clean document-level IDs after all chunk outputs are combined.
    events = []
    for number, item in enumerate(merged, start=1):
        event = item["event"]
        event.event_id = f"event_{number:02d}"
        events.append(event)

    return DocumentExtraction(document_id=document_id, events=events, notes=notes)


### Chunk context preflight

The same generated chunks are used by both workflows. This conservative check includes the prompts, schemas, a 4096-token response allowance, and an enlarged reserve for the Workflow B hazard catalogue.

In [ ]:
# Associate each task with its response schema for the context-size check.
SCHEMA_BY_TASK = {
    "workflow_a": DocumentExtraction,
    "hazards": HazardTaskOutput,
    "locations": LocationTaskOutput,
    "times": TimeTaskOutput,
    "impacts": ImpactTaskOutput,
}
MESSAGE_OVERHEAD_RESERVE = 2_000
HAZARD_CATALOGUE_BYTES_PER_TOKEN = 8
HAZARD_CATALOGUE_RESERVE = MAX_OUTPUT_TOKENS * HAZARD_CATALOGUE_BYTES_PER_TOKEN


def utf8_bytes(value):
    # A byte-level tokenizer cannot use more tokens than input bytes.
    return len(value.encode("utf-8"))


# Build a conservative bound for every chunk and workflow call. 
# The estimate reserves space for instructions, schemas, output, and Workflow B's catalogue.
empty_catalogue = HazardTaskOutput(document_id="preflight", events=[])
preflight_rows = []
for document in documents:
    document_id = document["document_id"]
    for chunk in document["chunks"]:
        text = chunk["text"]
        workflow_a_bound = (
            utf8_bytes(SYSTEM_PROMPT + build_workflow_a_prompt(document_id, text))
            + utf8_bytes(json.dumps(DocumentExtraction.model_json_schema(), ensure_ascii=False))
            + MESSAGE_OVERHEAD_RESERVE
            + MAX_OUTPUT_TOKENS
        )

        b_bounds = [
            utf8_bytes(SYSTEM_PROMPT + build_hazard_prompt(document_id, text))
            + utf8_bytes(json.dumps(HazardTaskOutput.model_json_schema(), ensure_ascii=False))
            + MESSAGE_OVERHEAD_RESERVE
            + MAX_OUTPUT_TOKENS
        ]
        for field_name, output_schema in [
            ("locations", LocationTaskOutput),
            ("times", TimeTaskOutput),
            ("impacts", ImpactTaskOutput),
        ]:
            prompt = build_enrichment_prompt(
                field_name, document_id, text, empty_catalogue
            )
            b_bounds.append(
                utf8_bytes(SYSTEM_PROMPT + prompt)
                + utf8_bytes(json.dumps(output_schema.model_json_schema(), ensure_ascii=False))
                + MESSAGE_OVERHEAD_RESERVE
                + HAZARD_CATALOGUE_RESERVE
                + MAX_OUTPUT_TOKENS
            )

        largest_bound = max(workflow_a_bound, *b_bounds)
        preflight_rows.append({
            "document_id": document_id,
            "chunk_id": chunk["chunk_id"],
            "words": chunk["word_count"],
            "largest_conservative_token_bound": largest_bound,
            "fits_128k_context": largest_bound <= CONTEXT_LIMIT,
        })

preflight = pd.DataFrame(preflight_rows)
display(preflight)
assert preflight["fits_128k_context"].all(), "Stop: at least one chunk does not safely fit."
print(f"All {len(preflight)} chunks pass the conservative context preflight.")

,document_id,chunk_id,words,largest_conservative_token_bound,fits_128k_context
0,PWC 2015 - Updated estimates on cost of Storm ...,chunk_001,484,44831,True
1,Krausmann 2014 - STREST report on lessons lear...,chunk_001,778,47675,True
2,Krausmann 2014 - STREST report on lessons lear...,chunk_002,791,47056,True
3,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,chunk_001,763,46654,True
4,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,chunk_002,800,46683,True
5,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,chunk_003,749,46433,True
6,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,chunk_004,765,46637,True
7,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,chunk_005,797,46876,True
8,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,chunk_006,800,46815,True
9,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,chunk_007,764,46489,True


All 52 chunks pass the conservative context preflight.


## 6. Run experiment

The final chunked experiment is disabled by default. When `RUN_LLM=True`, outputs are written under `data/final_results/`; the original pilot and feasibility results are never overwritten. Chunk responses and Workflow B intermediate responses are retained for later evaluation and error analysis.

In [10]:
# Save every validated Pydantic result as readable UTF-8 JSON.
def save_model(path, model):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(model.model_dump_json(indent=2) + "\n", encoding="utf-8")


# Run both workflows over the same documents and exactly the same chunks.
def run_final_experiment(llm, experiment_documents, results_dir=RESULTS_DIR):
    workflow_a_dir = results_dir / "workflow_a"
    workflow_b_dir = results_dir / "workflow_b"
    summary_path = results_dir / "experiment_summary.csv"
    protected_outputs = [workflow_a_dir, workflow_b_dir, summary_path]
    existing_outputs = [path for path in protected_outputs if path.exists()]
    # Stop before any API call if final results already exist.
    if existing_outputs:
        existing_text = ", ".join(str(path) for path in existing_outputs)
        raise FileExistsError(f"Existing final experiment results found: {existing_text}")

    summary_rows = []
    for number, document in enumerate(experiment_documents, start=1):
        document_key = f"document_{number:02d}"
        document_id = document["document_id"]
        chunks = document["chunks"]
        a_calls = 0
        b_calls = 0
        a_failures = []
        b_failures = []

        print(f"[{number}/9] {document_id}", flush=True)

        # Workflow A makes one complete extraction request for each chunk.
        a_chunk_outputs = []
        a_start = time.perf_counter()
        for chunk in chunks:
            print(f"  A {chunk['chunk_id']}", flush=True)
            a_calls += 1
            try:
                result = run_workflow_a(llm, document_id, chunk["text"])
                save_model(
                    workflow_a_dir / document_key / f"{chunk['chunk_id']}.json",
                    result,
                )
                a_chunk_outputs.append(result)
            except Exception as error:
                a_failures.append(
                    f"{chunk['chunk_id']}: {type(error).__name__}: {error}"
                )

        if not a_failures:
            try:
                result_a = merge_chunk_extractions(document_id, a_chunk_outputs)
                save_model(workflow_a_dir / document_key / "final.json", result_a)
            except Exception as error:
                a_failures.append(f"document merge: {type(error).__name__}: {error}")
        a_duration_seconds = time.perf_counter() - a_start

        # Workflow B makes one hazard call and three enrichment calls per chunk.
        # Failures are recorded as outcomes and are not retried.
        b_chunk_outputs = []
        b_start = time.perf_counter()
        for chunk in chunks:
            chunk_id = chunk["chunk_id"]
            chunk_dir = workflow_b_dir / document_key / chunk_id
            text = chunk["text"]
            outputs = {}

            print(f"  B {chunk_id} hazards", flush=True)
            b_calls += 1
            try:
                hazards = ask_for_structure(
                    llm, HazardTaskOutput, build_hazard_prompt(document_id, text)
                )
                outputs["hazards"] = hazards
                save_model(chunk_dir / "01_hazards.json", hazards)
            except Exception as error:
                b_failures.append(
                    f"{chunk_id} hazards: {type(error).__name__}: {error}"
                )

            if "hazards" not in outputs:
                continue

            stages = [
                (
                    "locations",
                    LocationTaskOutput,
                    build_location_prompt,
                    "02_locations.json",
                ),
                ("times", TimeTaskOutput, build_time_prompt, "03_times.json"),
                ("impacts", ImpactTaskOutput, build_impact_prompt, "04_impacts.json"),
            ]
            for name, output_schema, prompt_builder, filename in stages:
                print(f"  B {chunk_id} {name}", flush=True)
                b_calls += 1
                try:
                    result = ask_for_structure(
                        llm,
                        output_schema,
                        prompt_builder(document_id, text, outputs["hazards"]),
                    )
                    outputs[name] = result
                    save_model(chunk_dir / filename, result)
                except Exception as error:
                    b_failures.append(
                        f"{chunk_id} {name}: {type(error).__name__}: {error}"
                    )

            if all(
                name in outputs for name in ("hazards", "locations", "times", "impacts")
            ):
                try:
                    chunk_final = merge_decomposed_outputs(
                        outputs["hazards"],
                        outputs["locations"],
                        outputs["times"],
                        outputs["impacts"],
                    )
                    save_model(chunk_dir / "final.json", chunk_final)
                    b_chunk_outputs.append(chunk_final)
                except Exception as error:
                    b_failures.append(
                        f"{chunk_id} merge: {type(error).__name__}: {error}"
                    )

        if not b_failures:
            try:
                result_b = merge_chunk_extractions(document_id, b_chunk_outputs)
                save_model(workflow_b_dir / document_key / "final.json", result_b)
            except Exception as error:
                b_failures.append(f"document merge: {type(error).__name__}: {error}")
        b_duration_seconds = time.perf_counter() - b_start

        # Record completion, logical calls, runtime, and any failure messages.
        summary_rows.append({
            "document_id": document_id,
            "document_word_count": document["word_count"],
            "chunk_count": len(chunks),
            "workflow_a_status": "ok" if not a_failures else "failed",
            "workflow_b_status": "ok" if not b_failures else "failed",
            "workflow_a_logical_calls": a_calls,
            "workflow_b_logical_calls": b_calls,
            "workflow_a_duration_seconds": a_duration_seconds,
            "workflow_b_duration_seconds": b_duration_seconds,
            "workflow_a_failure_reason": " | ".join(a_failures),
            "workflow_b_failure_reason": " | ".join(b_failures),
        })
        results_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(summary_rows).to_csv(
            summary_path, index=False, encoding="utf-8"
        )

    return pd.DataFrame(summary_rows)


# This single safety switch controls whether paid API calls are made.
if RUN_LLM:
    summary = run_final_experiment(create_experiment_llm(), documents)
else:
    existing_summary = RESULTS_DIR / "experiment_summary.csv"
    if existing_summary.exists():
        summary = pd.read_csv(existing_summary)
        print("Loaded existing final experiment summary; no API calls were made.")
    else:
        summary = pd.DataFrame([
            {
                "document_id": document["document_id"],
                "document_word_count": document["word_count"],
                "chunk_count": len(document["chunks"]),
                "workflow_a_status": "not run (RUN_LLM=False)",
                "workflow_b_status": "not run (RUN_LLM=False)",
                "workflow_a_logical_calls": 0,
                "workflow_b_logical_calls": 0,
                "workflow_a_duration_seconds": pd.NA,
                "workflow_b_duration_seconds": pd.NA,
                "workflow_a_failure_reason": "",
                "workflow_b_failure_reason": "",
            }
            for document in documents
        ])
        print("Final LLM experiment disabled; no API calls were made.")

Loaded existing final experiment summary; no API calls were made.


## 7. Experiment summary

The summary reports document and chunk sizes, workflow status, logical calls, duration, and any failure reason. Workflow A normally uses one logical call per chunk and Workflow B uses four.

In [11]:
# Display the saved experiment status without recalculating extraction.
display(summary)

,document_id,document_word_count,chunk_count,workflow_a_status,workflow_b_status,workflow_a_logical_calls,workflow_b_logical_calls,workflow_a_duration_seconds,workflow_b_duration_seconds,workflow_a_failure_reason,workflow_b_failure_reason
0,PWC 2015 - Updated estimates on cost of Storm ...,484,1,ok,ok,1,4,14.516403,17.334018,NaN,NaN
1,Krausmann 2014 - STREST report on lessons lear...,1469,2,ok,ok,2,8,52.745652,117.539394,NaN,NaN
2,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,10477,16,failed,ok,16,64,844.779617,793.040045,chunk_005: ValueError: Model returned no valid...,NaN
3,Lavenir 2021 - À la recherche d'une jeune fill...,157,1,ok,ok,1,4,7.240332,9.882592,NaN,NaN
4,ABC 2024 - Traffic jams and flight delays due ...,706,1,ok,ok,1,4,26.660682,74.354174,NaN,NaN
5,"EFE 2024 - The DANA storm, live_ The death tol...",3635,6,failed,ok,6,24,394.892131,362.076217,chunk_001: ValueError: Model returned no valid...,NaN
6,Plag 2014 - Foreword extreme geohazards—a grow...,1457,3,ok,ok,3,12,7.059434,55.495937,NaN,NaN
7,Kadir 2014 - The Impact of Natural Disasters o...,4935,8,failed,failed,8,29,142.328956,402.289894,chunk_002: ValueError: Model returned no valid...,chunk_002 hazards: ValueError: Model returned ...
8,"Cavaleri 2019 - The October 29, 2018 storm in ...",9218,14,failed,failed,14,53,627.073231,1029.801941,chunk_009: RuntimeError: Model response was tr...,chunk_005 hazards: ValueError: Model returned ...
